In [1]:
# import importlib.util
# from pathlib import Path

# def _import_uatr_path():
#     seeds = []
#     p = globals().get("__vsc_ipynb_file__")
#     if isinstance(p, str):
#         seeds.append(Path(p).resolve().parent)
#     cwd = Path.cwd().resolve()
#     seeds.extend([cwd, *cwd.parents])
#     for base in seeds:
#         for cand in (
#             base / "_uatr_path.py",
#             base / "dev" / "research_nbs" / "_uatr_path.py",
#         ):
#             if cand.is_file():
#                 return cand
#     raise RuntimeError(
#         "Cannot find _uatr_path.py. Save/pull the repo and open the UATR folder in the editor."
#     )

# _spec = importlib.util.spec_from_file_location("_uatr_path", _import_uatr_path())
# _up = importlib.util.module_from_spec(_spec)
# assert _spec.loader is not None
# _spec.loader.exec_module(_up)
# _up.ensure_repo_on_sys_path()

import utils as ut

In [2]:
# imports and settings

import os
import time
import pickle
import warnings
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from copy import deepcopy

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

%load_ext autoreload
%autoreload 2

# do not show warnings
warnings.filterwarnings("ignore")

print("Imports complete.")

Imports complete.


In [3]:
path = '/Users/itamarmerfeld/Projects/UATR/data/ds2/pos/dpv1_1m.wav'
fs, data = wavfile.read(path)

In [4]:
# parameters and processing loop
height = 800
width = 1400
font_size = 16
fft_nperseg = 16000
percent_overlap = 0.25
window = 'hamming'
remove_dc = 20
crop_freq = 4000  # Hz, None to disable
normalization_window_size = 9  # in samples, None to disable
Q = 100

f_list = [636, 588, 900, 2000]

In [5]:
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

fig = make_subplots(rows=3, cols=len(f_list), column_titles=[f"f={f} Hz" for f in f_list], row_titles=["Phase Dist.", "Trans. Size Dist.", "Trans. Dist.", "S2G Graph"])
for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]

    # distribution of delta phases
    delta_phase = np.diff(f_phase)
    delta_phase = np.abs(delta_phase)
    hist_delta, bin_edges_delta = np.histogram(delta_phase, bins=Q, range=(0, 2 * np.pi), density=True)
    fig.add_trace(go.Bar(x=bin_edges_delta[:-1], y=hist_delta, name='Delta Phase Distribution', marker_color='orange', showlegend=False), row=1, col=ix)

    # distribution of traqnsitions
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=2, col=ix)

    # graph from S2G
    G = ut.get_s2g_graph(f_phase, n_levels=Q)
    fig_g = ut.draw_graph(G)
    for tr in fig_g.data:
                fig.add_trace(deepcopy(tr), row=3, col=ix)

fig.update_layout(height=1000, width=1000, title_text='Phase Distribution, Phase Transition-Size Distribution, Phase Transition Distribution, and S2G Graph for Selected Frequencies', font=dict(size=12))
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis.pdf")
fig.show()

In [6]:
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc, crop_freq=2000)
pxx = ut.calc_welch_from_spectrogram(Sxx, normalization_window_size=normalization_window_size)

fig = make_subplots(rows=1, cols=2, column_widths=[0.8, 0.2], column_titles=["Spectrogram", "Normalized Spectrum"], shared_yaxes=True)
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pxx, y=F, mode='lines', name='Normalized Spectrum', marker_color='Blue', showlegend=False), row=1, col=2)
fig.update_layout(height=600, width=1000, font=dict(size=12))
fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/frequency_analysis.svg")
fig.show()

In [61]:
f_list = [636, 588, 800, 2000]
Q=10
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

fig = make_subplots(rows=5, cols=len(f_list), column_titles=[f"f={f} Hz" for f in f_list], row_titles=["Phase Dist.", "Trans. Size Dist.", "Trans. Dist.", "S2G Graph, Eigenvalues", "projection"])
for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]

    # distribution of delta phases
    delta_phase = np.diff(f_phase)
    delta_phase = np.abs(delta_phase)
    hist_delta, bin_edges_delta = np.histogram(delta_phase, bins=Q, range=(0, 2 * np.pi), density=True)
    fig.add_trace(go.Bar(x=bin_edges_delta[:-1], y=hist_delta, name='Delta Phase Distribution', marker_color='orange', showlegend=False), row=1, col=ix)

    # distribution of traqnsitions
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    # x_transitions = ut.get_s2g(delta_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=2, col=ix)

    # graph from S2G
    G = ut.get_s2g_graph(f_phase, n_levels=Q)
    fig_g = ut.draw_graph(G)
    for tr in fig_g.data:
                fig.add_trace(deepcopy(tr), row=3, col=ix)

    # graph eigenvalues
    D = np.diag(np.sum(x_transitions, axis=1))
    L = D - x_transitions
    eigenvalues = LA.eigvals(L)
    eigenvalues = np.real(eigenvalues)
    eigenvalues = np.sort(eigenvalues)
    n = 10
    fig.add_trace(go.Scatter(x=np.arange(n), y=eigenvalues[:n], mode='lines', name='Eigenvalues', marker_color='Green', showlegend=False), row=4, col=ix)

    # projection
    A = (x_transitions + x_transitions.T) / 2
    D = np.diag(np.sum(A, axis=1))
    L = D - A
    vals, vecs = LA.eig(L)
    vals = np.real(vals)
    vals = np.sort(vals)
    vecs = np.real(vecs)
    vecs = np.sort(vecs, axis=1)
    x = vecs[:, 1]
    y = vecs[:, 2]

    coords = np.column_stack((x, y))
    centroid = np.mean(coords, axis=0)
    distances = np.linalg.norm(coords - centroid, axis=1)
    mean_dist = np.mean(distances)
    std_dist = np.std(distances)
    cv = std_dist / mean_dist

    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name='Eigenvectors', marker_color='Red', showlegend=False), row=5, col=ix)
    print(f"f={f}, cv={cv.round(3)}, gap={vals[2] - vals[1]}")

fig.update_layout(height=800, width=1000)
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis.pdf")
fig.show()

f=636, cv=0.5, gap=1.818512909999214
f=588, cv=0.477, gap=5.643003225462358
f=800, cv=0.572, gap=9.456875969415321
f=2000, cv=0.543, gap=1.887063237794763


In [39]:
G = nx.watts_strogatz_graph(n=100, k=4, p=0.1) # Small-world (Fuzzy Ring)
L = nx.laplacian_matrix(G).astype(np.float32).toarray()
vals, vecs = LA.eig(L)
x = vecs[:, 1]
y = vecs[:, 2]

fig = make_subplots(rows=1, cols=3)
# fig = go.Figure(data=[go.Scatter(x=x, y=y, mode='markers')])
fig.add_trace(go.Heatmap(z=L, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_traces(ut.draw_graph(G).data, rows=1, cols=2)
fig.add_trace(go.Scatter(x=x, y=y, mode='markers'), row=1, col=3)
fig.show(width=1000, height=200)


In [74]:
f = 588
f_ix = np.argmin(np.abs(F - f))
f_phase = phase[f_ix, :]

# distribution of delta phases
delta_phase = np.diff(f_phase)
delta_phase = np.abs(delta_phase)
hist_delta, bin_edges_delta = np.histogram(delta_phase, bins=10, range=(0, 2 * np.pi), density=True)
fig.add_trace(go.Bar(x=bin_edges_delta[:-1], y=hist_delta, name='Delta Phase Distribution', marker_color='orange', showlegend=False), row=1, col=ix)

# distribution of traqnsitions
x_transitions = ut.get_s2g(f_phase, n_levels=10)
x_transitions

array([[ 1,  0,  1,  2,  3,  5,  7,  9, 15,  4],
       [ 6,  5,  2,  3,  0,  2,  4, 14, 14, 22],
       [19, 13,  8,  0,  1,  2,  3,  0,  9, 14],
       [12, 21, 11,  4,  2,  0,  1,  1,  2,  7],
       [ 6, 14, 18,  6,  4,  1,  2,  0,  0,  7],
       [ 2,  7, 18, 11,  4,  3,  2,  1,  1,  5],
       [ 0,  7,  8, 22, 14, 13,  8,  2,  1,  5],
       [ 0,  3,  1,  7, 19, 14, 16,  5,  2,  0],
       [ 0,  1,  1,  3,  7, 11, 19, 13,  6,  1],
       [ 1,  1,  1,  3,  4,  3, 18, 21, 12,  4]])

In [63]:

def calculate_circularity_score(A):
    """
    Computes a circularity score for a transition/adjacency matrix.
    
    Score 1.0: Perfect ring/cycle.
    Score 0.5 - 0.8: Fuzzy ring (your first matrix).
    Score < 0.4: Dense noise or hub-and-spoke (your second matrix).
    """
    # 1. Symmetrize and handle diagonal (self-loops)
    A = np.array(A, dtype=float)
    np.fill_diagonal(A, 0) # Self-loops don't contribute to ring topology
    A_sym = (A + A.T) / 2
    
    # 2. Compute Laplacian L = D - A
    D = np.diag(np.sum(A_sym, axis=1))
    L = D - A_sym
    
    # 3. Eigen-decomposition (using eigh for symmetric matrices)
    vals, vecs = LA.eigh(L)
    
    # Skip the first eigenvalue (always 0 for connected graphs)
    l1, l2, l3 = vals[1], vals[2], vals[3]
    v1, v2 = vecs[:, 1], vecs[:, 2]
    
    # --- METRIC A: Spectral Symmetry ---
    # In a ring, l1 and l2 should be near-identical and significantly < l3
    # We penalize the gap between l1 and l2
    spectral_gap_penalty = abs(l1 - l2) / l3
    
    # --- METRIC B: Geometric Circularity ---
    # Project nodes into 2D and measure distance from centroid
    coords = np.column_stack((v1, v2))
    centroid = np.mean(coords, axis=0)
    distances = np.linalg.norm(coords - centroid, axis=1)
    
    # CV = Standard Deviation / Mean. Low CV = very circular.
    cv_radial = np.std(distances) / np.mean(distances)
    
    # --- COMPOSITE SCORE ---
    # We invert the penalties. 1.0 is perfect.
    # We weight the geometric CV more heavily as it detects "hubs" better.
    score = (1 - spectral_gap_penalty) * 0.3 + (1 - cv_radial) * 0.7
    final_score = max(0, min(1, score))
    return final_score, spectral_gap_penalty, cv_radial

# Testing the logic against your observations:
# Matrix 1 (The wrap-around): ~0.72 (Fuzzy Ring)
# Matrix 2 (The Node 2 Hub):  ~0.38 (Non-Circular)

In [76]:
f_list = [636, 588, 100, 2000]
Q=10
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

fig = make_subplots(rows=1, cols=len(f_list), column_titles=[f"f={f} Hz" for f in f_list], row_titles=["Trans. Dist."])
for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]

    # distribution of traqnsitions
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=1, col=ix)

    final_score, spectral_gap_penalty, cv_radial = calculate_circularity_score(x_transitions)
    print(f"f={f} Hz, score={final_score.round(3)}, gap={spectral_gap_penalty.round(3)}, cv={cv_radial.round(3)}")
fig.update_layout(height=300, width=800)
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis.pdf")
fig.show()

f=636 Hz, score=0.728, gap=0.031, cv=0.376
f=588 Hz, score=0.398, gap=0.098, cv=0.818
f=100 Hz, score=0.116, gap=0.226, cv=1.167
f=2000 Hz, score=0.323, gap=0.032, cv=0.953


In [84]:

def calculate_index_agnostic_circularity(A):
    """
    Determines circularity regardless of node ordering or stride.
    Works by recovering the 'natural' manifold order via polar sorting.
    """
    A = np.array(A, dtype=float)
    np.fill_diagonal(A, 0)
    
    # 1. Symmetrize
    A_sym = (A + A.T) / 2
    
    # 2. Use Normalized Laplacian (robust for varying weights/densities)
    d = np.sum(A_sym, axis=1)
    d_inv_sqrt = np.power(d, -0.5, where=d>0)
    D_inv_sqrt = np.diag(d_inv_sqrt)
    L_norm = np.eye(len(A)) - D_inv_sqrt @ A_sym @ D_inv_sqrt
    
    # 3. Eigen-decomposition
    vals, vecs = LA.eigh(L_norm)
    
    # Coordinates in the 2D manifold (Eigenvectors 1 and 2)
    v1, v2 = vecs[:, 1], vecs[:, 2]
    
    # 4. Metric A: Radial Consistency (Is it a hollow ring?)
    # All nodes in a ring should be roughly equidistant from the centroid
    coords = np.column_stack((v1, v2))
    centroid = np.mean(coords, axis=0)
    radii = np.linalg.norm(coords - centroid, axis=1)
    radial_cv = np.std(radii) / np.mean(radii)
    
    # 5. Metric B: Angular Distribution (Are nodes spread out?)
    # We sort by angle and check for large gaps
    angles = np.arctan2(v2, v1)
    sorted_angles = np.sort(angles)
    # Calculate the gaps between adjacent nodes around the circle
    gaps = np.diff(sorted_angles)
    # Add the wrap-around gap (last to first)
    gaps = np.append(gaps, (2 * np.pi - (sorted_angles[-1] - sorted_angles[0])))
    
    # If nodes are clustered in one spot (a hub), the max gap will be huge.
    # In a ring, gaps should be roughly 2*pi / N.
    expected_gap = (2 * np.pi) / len(A)
    max_gap_penalty = np.max(gaps) / (2 * np.pi)
    
    # 6. Composite Score
    # We prioritize Radial Consistency (0.7) and Angular Spread (0.3)
    circularity = (1 - radial_cv) * 0.7 + (1 - max_gap_penalty) * 0.3
    
    # Bonus: Reorder the matrix for inspection
    natural_order = np.argsort(angles)
    
    return round(max(0, circularity), 4), radial_cv, max_gap_penalty

# Verification with your Stride 8 matrix:
# score, order = calculate_index_agnostic_circularity(stride_8_matrix)
# print(f"Circularity: {score}")
# print(f"Discovered Sequence: {order}")

In [85]:
f_list = [636, 588, 800, 2000]
Q=10
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

fig = make_subplots(rows=1, cols=len(f_list), column_titles=[f"f={f} Hz" for f in f_list], row_titles=["Trans. Dist."])
for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]

    # distribution of traqnsitions
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=1, col=ix)

    final_score, radial_cv, max_gap_penalty = calculate_index_agnostic_circularity(x_transitions)
    print(f"f={f} Hz, score={final_score.round(3)}, radial_cv={radial_cv.round(3)}, max_gap_penalty={max_gap_penalty.round(3)}")
fig.update_layout(height=300, width=800)
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis.pdf")
fig.show()

f=636 Hz, score=0.737, radial_cv=0.284, max_gap_penalty=0.215
f=588 Hz, score=0.654, radial_cv=0.397, max_gap_penalty=0.226
f=800 Hz, score=0.785, radial_cv=0.197, max_gap_penalty=0.256
f=2000 Hz, score=0.606, radial_cv=0.467, max_gap_penalty=0.225


In [91]:
def calculate_heuristic_circularity(A, band_width=None):
    """
    Refactored Circularity Score based on:
    1. Shifting Peak (Stride detection and consistency)
    2. Banded Structure (Energy concentration)
    3. Symmetry & Decay (Weight drop-off from peak)
    """
    A = np.array(A, dtype=float)
    N = len(A)
    if N < 3: return 0.0
    
    # --- 1. SHIFTING PEAK (Stride Discovery) ---
    # Find the peak connection index for each row
    peaks = np.argmax(A, axis=1)
    
    # Calculate the strides (offsets) between consecutive rows
    # stride = (peak_curr - peak_prev) % N
    strides = np.diff(peaks) % N
    # Include wrap-around stride from last row to first
    strides = np.append(strides, (peaks[0] - peaks[-1]) % N)
    
    # Identify the dominant stride (the "mode")
    unique_strides, counts = np.unique(strides, return_counts=True)
    dominant_stride = unique_strides[np.argmax(counts)]
    stride_consistency = np.max(counts) / N
    
    # --- 2. BANDED STRUCTURE (Energy Concentration) ---
    # We measure how much weight is concentrated near the predicted peak
    total_energy = np.sum(A)
    if total_energy == 0: return 0.0
    
    banded_energy = 0
    # Window size for the band (e.g., 3 nodes: peak and its two neighbors)
    if band_width is None:
        window = [-1, 0, 1] 
    else:
        window = np.arange(-band_width, band_width + 1)
    
    for i in range(N):
        # The predicted peak for row i based on the dominant stride
        # (Assuming Row 0's peak is the starting point)
        predicted_peak = (peaks[0] + i * dominant_stride) % N
        
        for w in window:
            neighbor_idx = (predicted_peak + w) % N
            banded_energy += A[i, neighbor_idx]
            
    banded_ratio = banded_energy / total_energy
    
    # --- 3. SYMMETRY & DECAY ---
    # Check if values decrease as we move away from the peak index
    decay_violations = 0
    total_comparisons = 0
    
    for i in range(N):
        p = peaks[i]
        # Check 2 steps out in both directions
        # Peak should be > neighbors, neighbors should be > their neighbors
        for dist in range(1, 3):
            left = (p - dist) % N
            right = (p + dist) % N
            inner_left = (p - (dist - 1)) % N
            inner_right = (p + (dist - 1)) % N
            
            # Penalize if further nodes have higher values than inner nodes
            if A[i, left] > A[i, inner_left]: decay_violations += 1
            if A[i, right] > A[i, inner_right]: decay_violations += 1
            total_comparisons += 2
            
    decay_score = 1 - (decay_violations / total_comparisons)
    
    # --- FINAL SCORE COMPOSITION ---
    # 0.4 Weight on Stride Consistency (The "Circular" loop logic)
    # 0.3 Weight on Bandedness (The "Fuzzy" ring thickness)
    # 0.3 Weight on Decay (The signal-to-noise ratio of the ring)
    
    final_score = (stride_consistency * 0.4) + (banded_ratio * 0.3) + (decay_score * 0.3)
    
    return {
        "circularity_score": round(final_score, 4),
        "detected_stride": int(dominant_stride),
        "stride_consistency": round(stride_consistency, 2),
        "banded_ratio": round(banded_ratio, 2),
        "decay_score": round(decay_score, 2)
    }

# Example usage with your Stride 8 matrix:
# results = calculate_heuristic_circularity(stride_8_matrix)
# print(results)

In [92]:
f_list = [636, 588, 800, 2000]
Q=100
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

fig = make_subplots(rows=1, cols=len(f_list), column_titles=[f"f={f} Hz" for f in f_list], row_titles=["Trans. Dist."])
for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]

    # distribution of traqnsitions
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=1, col=ix)

    score_dict = calculate_heuristic_circularity(x_transitions, band_width=10)
    circularity_score = score_dict["circularity_score"]
    detected_stride = score_dict["detected_stride"]
    stride_consistency = score_dict["stride_consistency"]
    banded_ratio = score_dict["banded_ratio"]
    decay_score = score_dict["decay_score"]
    print(f"f={f} Hz, score={circularity_score}, stride={detected_stride}, consistency={stride_consistency}, banded={banded_ratio}, decay={decay_score}")
fig.update_layout(height=300, width=800)
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis.pdf")
fig.show()

f=636 Hz, score=0.393, stride=5, consistency=0.1, banded=0.22, decay=0.95
f=588 Hz, score=0.3708, stride=14, consistency=0.04, banded=0.21, decay=0.97
f=800 Hz, score=0.37, stride=3, consistency=0.05, banded=0.2, decay=0.97
f=2000 Hz, score=0.3784, stride=94, consistency=0.07, banded=0.19, decay=0.98


In [99]:
from research_nbs.check_circle_v2 import transition_matrix_circularity

F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc, crop_freq=2000)
pxx = ut.calc_welch_from_spectrogram(Sxx, normalization_window_size=normalization_window_size)
circularity_score = []
circularity_period = []
circularity_leading_eigenvalue = []
circularity_eigenvalues = []
circularity_spectral_concentration = []

for fi in range(len(F)):
    M = ut.get_s2g(phase[fi, :], n_levels=Q)
    circularity = transition_matrix_circularity(M)
    circularity_score.append(circularity.score)
    circularity_period.append(circularity.period)
    circularity_leading_eigenvalue.append(circularity.leading_eigenvalue)
    circularity_eigenvalues.append(circularity.eigenvalues)
    circularity_spectral_concentration.append(circularity.spectral_concentration)

fig = make_subplots(rows=1, cols=6, column_widths=[0.5, 0.1, 0.1, 0.1, 0.1, 0.1], column_titles=["Spectrogram", "Normalized Spectrum", "Circularity Score", "Circularity Period", "Circularity Leading Eigenvalue", "Circularity Eigenvalues"], shared_yaxes=True)
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pxx, y=F, mode='lines', name='Normalized Spectrum', marker_color='Blue', showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=circularity_score, y=F, mode='lines', name='Circularity Score', marker_color='Red', showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(x=circularity_spectral_concentration, y=F, mode='lines', name='Circularity Score', marker_color='Red', showlegend=False), row=1, col=4)
fig.add_trace(go.Scatter(x=np.abs(circularity_leading_eigenvalue), y=F, mode='lines', name='Circularity Score', marker_color='Red', showlegend=False), row=1, col=5)
fig.add_trace(go.Scatter(x=circularity_period, y=F, mode='lines', name='Circularity Score', marker_color='Red', showlegend=False), row=1, col=6)
fig.update_layout(height=600, width=1000, font=dict(size=12))
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/frequency_analysis.svg")
fig.show()